# Climate Trends Dashboard — South America (1950–2100)

MSc Data Science · Universidad ICESI · Davinson Arteaga & Gian Mendoza

Interactive analysis of temperature and relative humidity trends in South America using the NEX-GDDP-CMIP6 dataset (8 GCMs, 4 SSPs + historical). Each section explores one analytical piece; the final section consolidates the main components into a two-tab dashboard.

## Section 0 — Setup and data loading

In [2]:
import sys
import warnings
from pathlib import Path
from datetime import datetime
from functools import lru_cache

import numpy as np
import pandas as pd
import xarray as xr

import holoviews as hv
import hvplot.pandas  # noqa: F401
import hvplot.xarray  # noqa: F401
import panel as pn

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geoviews as gv
import geoviews.feature as gf

warnings.filterwarnings('ignore')
pn.extension('tabulator', sizing_mode='stretch_width')
hv.extension('bokeh')
gv.extension('bokeh')

sys.path.insert(0, str(Path.cwd()))
import analysis_lib as al

print(f'Notebook started · {datetime.now():%Y-%m-%d %H:%M:%S}')

Notebook started · 2026-06-11 02:59:17


In [3]:
BASE_DIR = Path('/media/volume/jay2vol/nexgddp')
PROC_DIR = BASE_DIR / 'processed'
SPATIAL_TRENDS_DIR = BASE_DIR / 'outputs' / '10_spatial_trends'

df_annual = pd.concat([
    pd.read_csv(PROC_DIR / 'annual_subregion_anomalies.csv'),
    pd.read_csv(PROC_DIR / 'annual_continental_anomalies.csv'),
], ignore_index=True)
df_monthly = pd.read_csv(PROC_DIR / 'monthly_subregion_anomalies.csv')

print(f'Annual: {len(df_annual):,} rows · Monthly: {len(df_monthly):,} rows')

Annual: 32,720 rows · Monthly: 314,112 rows


In [4]:
MODELS = sorted(df_annual['model'].unique())
SCENARIOS = sorted(df_annual['scenario'].unique())
SUBREGIONS = sorted(df_annual['subregion'].unique())
SUBREGIONS_MONTHLY = [s for s in SUBREGIONS if 'continental' not in s.lower()]
VARIABLES = sorted(df_annual['variable'].unique())
YEAR_MIN, YEAR_MAX = int(df_annual['year'].min()), int(df_annual['year'].max())

SSP_ORDER = ['historical', 'ssp126', 'ssp245', 'ssp370', 'ssp585']
SCENARIO_COLORS = {
    'historical': '#333333', 'ssp126': '#1f77b4', 'ssp245': '#daa520',
    'ssp370': '#ff7f0e', 'ssp585': '#d62728',
}
SCENARIO_LABELS = {
    'historical': 'Historical (1950–2014)',
    'ssp126': 'SSP1-2.6 (Sustainable)',
    'ssp245': 'SSP2-4.5 (Middle road)',
    'ssp370': 'SSP3-7.0 (Fragmentation)',
    'ssp585': 'SSP5-8.5 (Fossil-fueled)',
}
VARIABLE_LABELS = {'tas': 'Air temperature (tas)', 'hurs': 'Relative humidity (hurs)'}

print(f'Models: {len(MODELS)} · Scenarios: {len(SCENARIOS)} · Subregions: {len(SUBREGIONS)} · Range: {YEAR_MIN}–{YEAR_MAX}')

Models: 8 · Scenarios: 5 · Subregions: 5 · Range: 1950–2100


---

## Section 1 — Annual time series

Annual anomalies referenced to 1960–2014 climatology. Bands show p10–p90 across the selected GCM ensemble, recomputed on the fly.

### 1.1 Time series per subregion with uncertainty bands

In [5]:
w11_variable = pn.widgets.RadioBoxGroup(name='Variable', options={VARIABLE_LABELS[v]: v for v in VARIABLES}, value='tas', inline=True)
w11_subregion = pn.widgets.Select(name='Subregion', options=SUBREGIONS, value='Central Amazonia' if 'Central Amazonia' in SUBREGIONS else SUBREGIONS[0], width=260)
w11_scenarios = pn.widgets.CheckButtonGroup(name='Scenarios', options={SCENARIO_LABELS[s].split(' (')[0]: s for s in SSP_ORDER if s in SCENARIOS}, value=[s for s in SSP_ORDER if s in SCENARIOS], button_type='primary', button_style='outline')
w11_models = pn.widgets.MultiChoice(name='GCM ensemble', options=MODELS, value=MODELS, width=520)
w11_year_range = pn.widgets.IntRangeSlider(name='Time window', start=YEAR_MIN, end=YEAR_MAX, value=(YEAR_MIN, YEAR_MAX), step=1, width=520)
w11_show_band = pn.widgets.Checkbox(name='Show p10–p90 band', value=True)

In [6]:
def build_timeseries_plot(variable, subregion, scenarios, models, year_range, show_band):
    if not scenarios or not models:
        return pn.pane.Markdown('⚠️ Select at least one scenario and one model.')
    unit = al.VAR_UNITS[variable]['value']
    layers = []
    for scenario in scenarios:
        ens = al.build_annual_series_ensemble(df_annual=df_annual, scenario=scenario, variable=variable, subregion=subregion, year_range=year_range, selected_models=models, use_historical_before_2015=(scenario != 'historical'))
        if len(ens) == 0:
            continue
        if scenario == 'historical':
            ens = ens[ens['year'] <= 2014]
        color = SCENARIO_COLORS[scenario]
        if show_band:
            layers.append(ens.hvplot.area(x='year', y='ens_p10', y2='ens_p90', alpha=0.22, line_width=0, color=color, hover=False))
        layers.append(ens.hvplot.line(x='year', y='ens_p50', color=color, line_width=2, label=SCENARIO_LABELS[scenario].split(' (')[0], hover_cols=['n_models', 'ens_p10', 'ens_p90']))
    if not layers:
        return pn.pane.Markdown('⚠️ No data for current selection.')
    vline = hv.VLine(2015).opts(color='gray', line_dash='dashed', line_width=1)
    hline = hv.HLine(0).opts(color='gray', line_dash='dotted', line_width=1)
    return (hv.Overlay(layers) * vline * hline).opts(
        title=f'{VARIABLE_LABELS[variable]} · {subregion} (ref 1960–2014)',
        ylabel=f'Anomaly {variable} ({unit})', xlabel='Year',
        width=900, height=420, legend_position='bottom_right', show_grid=True, toolbar='above',
    )

timeseries_pane = pn.bind(build_timeseries_plot, variable=w11_variable, subregion=w11_subregion, scenarios=w11_scenarios, models=w11_models, year_range=w11_year_range, show_band=w11_show_band)

pn.Column(
    pn.Row(pn.Column('**Variable**', w11_variable, width=240), pn.Column('**Subregion**', w11_subregion, width=280), pn.Column('**Band**', w11_show_band, width=200)),
    pn.Column('**Scenarios**', w11_scenarios, '**GCM ensemble**', w11_models, '**Time window**', w11_year_range),
    pn.pane.Markdown('---'),
    timeseries_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'ffba1c72-63d3-4a7c-819f-d9cc0902288a': {'version…

### 1.2 Cross-subregion comparison

Median-only lines per subregion under a single scenario, to compare regional warming rates.

In [7]:
w12_variable = pn.widgets.RadioBoxGroup(name='Variable', options={VARIABLE_LABELS[v]: v for v in VARIABLES}, value='tas', inline=True)
w12_scenario = pn.widgets.Select(name='Scenario', options={SCENARIO_LABELS[s]: s for s in SSP_ORDER if s in SCENARIOS and s != 'historical'}, value='ssp585', width=320)
w12_models = pn.widgets.MultiChoice(name='GCM ensemble', options=MODELS, value=MODELS, width=520)
w12_year_range = pn.widgets.IntRangeSlider(name='Time window', start=YEAR_MIN, end=YEAR_MAX, value=(YEAR_MIN, YEAR_MAX), step=1, width=520)

SUBREGION_COLORS = {
    'Northern Tropics': '#e41a1c', 'Central Amazonia': '#377eb8',
    'Subtropical Andes': '#4daf4a', 'Southern Cone': '#984ea3',
    'South America (continental)': '#000000',
}

def build_subregion_comparison_plot(variable, scenario, models, year_range):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    unit = al.VAR_UNITS[variable]['value']
    layers = []
    for subregion in SUBREGIONS:
        ens = al.build_annual_series_ensemble(df_annual=df_annual, scenario=scenario, variable=variable, subregion=subregion, year_range=year_range, selected_models=models, use_historical_before_2015=True)
        if len(ens) == 0:
            continue
        color = SUBREGION_COLORS.get(subregion, '#888888')
        lw = 2.5 if 'continental' in subregion.lower() else 1.8
        layers.append(ens.hvplot.line(x='year', y='ens_p50', color=color, line_width=lw, label=subregion, hover_cols=['n_models']))
    if not layers:
        return pn.pane.Markdown('⚠️ No data.')
    vline = hv.VLine(2015).opts(color='gray', line_dash='dashed', line_width=1)
    hline = hv.HLine(0).opts(color='gray', line_dash='dotted', line_width=1)
    return (hv.Overlay(layers) * vline * hline).opts(
        title=f'Cross-subregion · {VARIABLE_LABELS[variable]} · {SCENARIO_LABELS[scenario]}',
        ylabel=f'Anomaly {variable} ({unit}) — ensemble median', xlabel='Year',
        width=900, height=420, legend_position='top_left', show_grid=True, toolbar='above',
    )

subregion_comparison_pane = pn.bind(build_subregion_comparison_plot, variable=w12_variable, scenario=w12_scenario, models=w12_models, year_range=w12_year_range)

pn.Column(
    pn.Row(pn.Column('**Variable**', w12_variable, width=240), pn.Column('**Scenario**', w12_scenario, width=340)),
    pn.Column('**GCM ensemble**', w12_models, '**Time window**', w12_year_range),
    pn.pane.Markdown('---'),
    subregion_comparison_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'2a6b2185-2651-4727-afe7-a7c3b780df79': {'version…

### 1.3 Inter-model spread

Each GCM as an individual thin line plus the ensemble median in black. Identifies outliers and confirms ensemble coherence.

In [8]:
w13_variable = pn.widgets.RadioBoxGroup(name='Variable', options={VARIABLE_LABELS[v]: v for v in VARIABLES}, value='tas', inline=True)
w13_subregion = pn.widgets.Select(name='Subregion', options=SUBREGIONS, value='Central Amazonia' if 'Central Amazonia' in SUBREGIONS else SUBREGIONS[0], width=280)
w13_scenario = pn.widgets.Select(name='Scenario', options={SCENARIO_LABELS[s]: s for s in SSP_ORDER if s in SCENARIOS and s != 'historical'}, value='ssp585', width=320)
w13_models = pn.widgets.MultiChoice(name='GCMs to show', options=MODELS, value=MODELS, width=520)
w13_year_range = pn.widgets.IntRangeSlider(name='Time window', start=YEAR_MIN, end=YEAR_MAX, value=(YEAR_MIN, YEAR_MAX), step=1, width=520)

MODEL_COLORS = {
    'ACCESS-CM2': '#1f77b4', 'CanESM5': '#ff7f0e', 'CESM2': '#2ca02c',
    'EC-Earth3': '#d62728', 'GFDL-ESM4': '#9467bd', 'IPSL-CM6A-LR': '#8c564b',
    'MIROC6': '#e377c2', 'MPI-ESM1-2-HR': '#7f7f7f',
}

def build_intermodel_spread_plot(variable, subregion, scenario, models, year_range):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    unit = al.VAR_UNITS[variable]['value']
    layers, series_for_median = [], []
    for model in models:
        years, vals = al.build_annual_series_per_model(df_annual=df_annual, model=model, scenario=scenario, variable=variable, subregion=subregion, year_range=year_range, use_historical_before_2015=True)
        if len(years) == 0:
            continue
        df_m = pd.DataFrame({'year': years, 'anomaly': vals, 'model': model})
        series_for_median.append(df_m)
        layers.append(df_m.hvplot.line(x='year', y='anomaly', color=MODEL_COLORS.get(model, '#cccccc'), line_width=1.0, alpha=0.75, label=model, hover_cols=['model']))
    if not layers:
        return pn.pane.Markdown('⚠️ No data.')
    df_all = pd.concat(series_for_median, ignore_index=True)
    df_median = df_all.groupby('year', as_index=False)['anomaly'].median()
    layers.append(df_median.hvplot.line(x='year', y='anomaly', color='black', line_width=3, label='Ensemble median'))
    vline = hv.VLine(2015).opts(color='gray', line_dash='dashed', line_width=1)
    hline = hv.HLine(0).opts(color='gray', line_dash='dotted', line_width=1)
    return (hv.Overlay(layers) * vline * hline).opts(
        title=f'Inter-model spread · {VARIABLE_LABELS[variable]} · {subregion} · {SCENARIO_LABELS[scenario]}',
        ylabel=f'Anomaly {variable} ({unit})', xlabel='Year',
        width=900, height=420, legend_position='top_left', show_grid=True, toolbar='above',
    )

intermodel_spread_pane = pn.bind(build_intermodel_spread_plot, variable=w13_variable, subregion=w13_subregion, scenario=w13_scenario, models=w13_models, year_range=w13_year_range)

pn.Column(
    pn.Row(pn.Column('**Variable**', w13_variable, width=240), pn.Column('**Subregion**', w13_subregion, width=300), pn.Column('**Scenario**', w13_scenario, width=340)),
    pn.Column('**GCMs**', w13_models, '**Time window**', w13_year_range),
    pn.pane.Markdown('---'),
    intermodel_spread_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'adb0be57-7b59-45c0-938d-06a125a40e60': {'version…

---

## Section 2 — Mann-Kendall + Sen's slope

Non-parametric trend detection with Hamed-Rao autocorrelation correction. Sen's slope reports trend magnitude per decade. Recomputed on-the-fly on the ensemble median.

In [9]:
w21_variable = pn.widgets.RadioBoxGroup(name='Variable', options={VARIABLE_LABELS[v]: v for v in VARIABLES}, value='tas', inline=True)
w21_window = pn.widgets.RadioBoxGroup(name='Window', options={'Historical 1950–2014': 'historical_1950_2014', 'Projected 2015–2100': 'projected_2015_2100', 'Full 1950–2100': 'full_1950_2100'}, value='full_1950_2100', inline=True)
w21_models = pn.widgets.MultiChoice(name='GCM ensemble', options=MODELS, value=MODELS, width=520)

In [10]:
WINDOW_SPECS = {
    'historical_1950_2014': {'hist': (1950, 2014), 'ssp': None,        'scenarios': ['historical']},
    'projected_2015_2100':  {'hist': None,         'ssp': (2015, 2100),'scenarios': ['ssp126', 'ssp245', 'ssp370', 'ssp585']},
    'full_1950_2100':       {'hist': (1950, 2014), 'ssp': (2015, 2100),'scenarios': ['ssp126', 'ssp245', 'ssp370', 'ssp585']},
}

def format_slope_cell(slope, pvalue):
    if not np.isfinite(slope) or not np.isfinite(pvalue):
        return ''
    marker = '**' if pvalue < 0.01 else ('*' if pvalue < 0.05 else '')
    sign = '+' if slope >= 0 else ''
    return f'{sign}{slope:.2f}{marker}'

def extract_numeric_slope(s):
    if not s:
        return np.nan
    try:
        return float(s.rstrip('*').strip())
    except ValueError:
        return np.nan

def compute_mk_table(variable, window_key, models):
    spec = WINDOW_SPECS[window_key]
    if spec['hist'] is not None and spec['ssp'] is not None:
        year_range = (spec['hist'][0], spec['ssp'][1])
    elif spec['hist'] is not None:
        year_range = spec['hist']
    else:
        year_range = spec['ssp']
    rows = []
    for subregion in SUBREGIONS:
        row = {'Subregion': subregion}
        for scenario in spec['scenarios']:
            ens = al.build_annual_series_ensemble(df_annual=df_annual, scenario=scenario, variable=variable, subregion=subregion, year_range=year_range, selected_models=models, use_historical_before_2015=(spec['hist'] is not None and spec['ssp'] is not None))
            if len(ens) < 10:
                row[scenario] = ''
                continue
            r = al.run_mk_full(ens['ens_p50'].values)
            row[scenario] = format_slope_cell(r['sen_slope_per_decade'], r['mk_hr_p'])
        rows.append(row)
    return pd.DataFrame(rows)

def build_mk_table_panel(variable, window_key, models):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    df_table = compute_mk_table(variable, window_key, models)
    numeric_cols = [c for c in df_table.columns if c != 'Subregion']
    all_vals = []
    for col in numeric_cols:
        all_vals.extend([extract_numeric_slope(v) for v in df_table[col].values])
    all_vals = [abs(v) for v in all_vals if np.isfinite(v)]
    abs_max = max(all_vals) if all_vals else 1.0
    def style_cell(val):
        slope = extract_numeric_slope(val)
        if not np.isfinite(slope):
            return ''
        intensity = min(abs(slope) / abs_max, 1.0)
        is_alarming = (slope > 0) if variable == 'tas' else (slope < 0)
        if is_alarming:
            r, g, b = 220, int(240 - 140 * intensity), int(240 - 200 * intensity)
        else:
            r, g, b = int(240 - 140 * intensity), int(240 - 60 * intensity), 240
        return f'background-color: rgb({r},{g},{b})'
    styler = df_table.style.map(style_cell, subset=numeric_cols).set_properties(**{'text-align': 'center', 'font-weight': '600'})
    units = al.VAR_UNITS[variable]['slope']
    window_label = {'historical_1950_2014': 'Historical 1950–2014', 'projected_2015_2100': 'Projected 2015–2100', 'full_1950_2100': 'Full 1950–2100'}[window_key]
    header = pn.pane.Markdown(f"**Sen's slope ({units}) · MK Hamed-Rao · {window_label}**  \nEnsemble: {len(models)} / {len(MODELS)} models · `**` p<0.01, `*` p<0.05")
    return pn.Column(header, pn.pane.DataFrame(styler, index=False, escape=False, sizing_mode='stretch_width', max_rows=20))

mk_table_pane = pn.bind(build_mk_table_panel, variable=w21_variable, window_key=w21_window, models=w21_models)

pn.Column(
    pn.Row(pn.Column('**Variable**', w21_variable, width=240), pn.Column('**Window**', w21_window, width=420)),
    pn.Column('**GCM ensemble**', w21_models),
    pn.pane.Markdown('---'),
    mk_table_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'8abb4194-c79b-4747-9029-962abf0a63c3': {'version…

---

## Section 3 — Hawkins & Sutton uncertainty decomposition

Splits projection variance into internal (I), model (M) and scenario (S) sources. T(t) = I + M(t) + S(t). Left plot: stacked fractions. Right plot: absolute components.

In [11]:
w31_variable = pn.widgets.RadioBoxGroup(name='Variable', options={VARIABLE_LABELS[v]: v for v in VARIABLES}, value='tas', inline=True)
w31_subregion = pn.widgets.Select(name='Subregion', options=SUBREGIONS, value='South America (continental)' if 'South America (continental)' in SUBREGIONS else SUBREGIONS[0], width=320)
w31_models = pn.widgets.MultiChoice(name='GCM ensemble', options=MODELS, value=MODELS, width=520)

HS_COLORS = {'I': '#d62728', 'M': '#1f77b4', 'S': '#2ca02c', 'T': '#222222'}

def build_hawkins_sutton_plots(variable, subregion, models):
    if len(models) < 2:
        return pn.pane.Markdown('⚠️ H&S requires at least 2 models.')
    df_hs = al.hawkins_sutton_decompose(df_annual=df_annual, variable=variable, subregion=subregion, selected_models=models, scenarios=list(al.SSPs), year_start=1950, year_end=2100, hs_ref_start=1995, hs_ref_end=2014, poly_degree=4)
    if len(df_hs) == 0:
        return pn.pane.Markdown('⚠️ Empty decomposition.')
    df_hs = df_hs[df_hs['year'] >= 2015].copy()
    df_frac = df_hs[['year', 'F_I', 'F_M', 'F_S']].copy()
    df_frac.columns = ['year', 'Internal (I)', 'Model (M)', 'Scenario (S)']
    df_long = df_frac.melt(id_vars='year', var_name='Source', value_name='Fraction')
    df_long['Source'] = pd.Categorical(df_long['Source'], categories=['Internal (I)', 'Model (M)', 'Scenario (S)'], ordered=True)
    df_long = df_long.sort_values(['year', 'Source'])
    fractions_plot = df_long.hvplot.area(x='year', y='Fraction', by='Source', stacked=True, color=[HS_COLORS['I'], HS_COLORS['M'], HS_COLORS['S']], alpha=0.85, line_width=0).opts(
        title=f'Variance fraction · {VARIABLE_LABELS[variable]} · {subregion}',
        ylabel='Fraction (F_I + F_M + F_S = 1)', xlabel='Year',
        width=470, height=380, legend_position='top_left', show_grid=True, toolbar='above', ylim=(0, 1),
    )
    unit_sq = '(°C)²' if variable == 'tas' else '(%)²'
    line_I = df_hs.hvplot.line(x='year', y='I', color=HS_COLORS['I'], line_width=2, label='I (internal)')
    line_M = df_hs.hvplot.line(x='year', y='M', color=HS_COLORS['M'], line_width=2, label='M (model)')
    line_S = df_hs.hvplot.line(x='year', y='S', color=HS_COLORS['S'], line_width=2, label='S (scenario)')
    line_T = df_hs.hvplot.line(x='year', y='T', color=HS_COLORS['T'], line_width=2, line_dash='dashed', label='T (total)')
    abs_plot = (line_I * line_M * line_S * line_T).opts(
        title=f'Absolute components · {unit_sq}',
        ylabel=f'Variance {unit_sq}', xlabel='Year',
        width=470, height=380, legend_position='top_left', show_grid=True, toolbar='above',
    )
    return pn.Row(pn.pane.HoloViews(fractions_plot), pn.pane.HoloViews(abs_plot))

hs_plots_pane = pn.bind(build_hawkins_sutton_plots, variable=w31_variable, subregion=w31_subregion, models=w31_models)

pn.Column(
    pn.Row(pn.Column('**Variable**', w31_variable, width=240), pn.Column('**Subregion**', w31_subregion, width=340)),
    pn.Column('**GCM ensemble**', w31_models),
    pn.pane.Markdown('---'),
    hs_plots_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'887f3171-c44f-40cf-a9ee-e2a683d2e8ae': {'version…

---

## Section 4 — Spatial trend maps

Interactive **GeoViews/Bokeh** spatial maps using the pre-computed NetCDF outputs from script 10. Left: Sen's slope. Right: inter-model sign agreement. Both maps include hover inspection and use a South America Plate Carrée view.

In [12]:
w41_variable = pn.widgets.RadioBoxGroup(
    name='Variable',
    options={VARIABLE_LABELS[v]: v for v in VARIABLES},
    value='tas',
    inline=True,
)
w41_window = pn.widgets.RadioBoxGroup(
    name='Window',
    options={
        'Historical 1950–2014': 'historical_1950_2014',
        'Projected 2015–2100': 'projected_2015_2100',
        'Full 1950–2100': 'full_1950_2100',
    },
    value='full_1950_2100',
    inline=True,
)
w41_scenario = pn.widgets.Select(
    name='SSP scenario',
    options={SCENARIO_LABELS[s]: s for s in ['ssp126', 'ssp245', 'ssp370', 'ssp585']},
    value='ssp585',
    width=340,
)

In [13]:
SA_EXTENT = (-82, -34, -56, 13)
WINDOW_LABELS = {
    'historical_1950_2014': '1950–2014',
    'projected_2015_2100': '2015–2100',
    'full_1950_2100': '1950–2100',
}


def _build_label(variable, window_key, scenario):
    if window_key == 'historical_1950_2014':
        return f'{variable}__{window_key}'
    return f'{variable}__{window_key}__{scenario}'


def _normalize_spatial_da(da):
    da = da.copy()
    if float(da['lon'].max()) > 180:
        da = da.assign_coords(lon=(((da['lon'] + 180) % 360) - 180))
    da = da.sortby('lon')
    if da['lat'].values[0] > da['lat'].values[-1]:
        da = da.sortby('lat')
    return da


def _open_spatial_dataarray(path, variable_name):
    with xr.open_dataset(path) as ds:
        da = ds[variable_name].load()
    return _normalize_spatial_da(da.rename(variable_name))


@lru_cache(maxsize=64)
def _load_spatial_fields(variable, window_key, scenario):
    label = _build_label(variable, window_key, scenario)
    slope_path = SPATIAL_TRENDS_DIR / f'sen_slope_{label}.nc'
    agree_path = SPATIAL_TRENDS_DIR / f'agreement_{label}.nc'

    missing = [p.name for p in [slope_path, agree_path] if not p.exists()]
    if missing:
        return None, missing

    slope = _open_spatial_dataarray(slope_path, 'sen_slope')
    agree = _open_spatial_dataarray(agree_path, 'agreement_pct')
    return (slope, agree), []


def _spatial_title_parts(variable, window_key, scenario):
    scen_label = 'Historical' if window_key == 'historical_1950_2014' else SCENARIO_LABELS.get(scenario, scenario).split(' (')[0]
    window_label = WINDOW_LABELS.get(window_key, window_key)
    variable_label = VARIABLE_LABELS.get(variable, variable)
    return variable_label, scen_label, window_label


def _safe_symmetric_clim(da, min_abs=0.05, percentile=98):
    values = da.values
    finite = np.isfinite(values)
    if not finite.any():
        return (-min_abs, min_abs)
    abs_max = float(np.nanpercentile(np.abs(values[finite]), percentile))
    abs_max = max(abs_max, min_abs)
    return (-abs_max, abs_max)


def _make_geoviews_image(da, value_dimension, cmap, clim, width, height, title, clabel):
    image = gv.Dataset(da, kdims=['lon', 'lat'], vdims=value_dimension).to(gv.Image, ['lon', 'lat'])
    return image.opts(
        cmap=cmap,
        colorbar=True,
        clim=clim,
        width=width,
        height=height,
        tools=['hover'],
        toolbar='above',
        title=title,
        clabel=clabel,
    )


def build_spatial_maps_panel(variable, window_key, scenario):
    loaded, missing = _load_spatial_fields(variable, window_key, scenario)
    if loaded is None:
        return pn.pane.Markdown(
            '⚠️ Spatial files are not available for this selection:\n\n' + '\n'.join(f'- `{name}`' for name in missing),
            sizing_mode='stretch_width',
        )

    slope, agree = loaded
    variable_label, scen_label, window_label = _spatial_title_parts(variable, window_key, scenario)
    slope_units = al.VAR_UNITS[variable]['slope']
    slope_cmap = 'RdBu_r' if variable == 'tas' else 'RdBu'

    slope_map = _make_geoviews_image(
        da=slope,
        value_dimension='sen_slope',
        cmap=slope_cmap,
        clim=_safe_symmetric_clim(slope),
        width=560,
        height=610,
        title=f"Sen's slope · {variable} · {scen_label} · {window_label}",
        clabel=slope_units,
    )
    agreement_map = _make_geoviews_image(
        da=agree,
        value_dimension='agreement_pct',
        cmap='YlGnBu',
        clim=(50, 100),
        width=560,
        height=610,
        title=f'Agreement · {variable} · {scen_label}',
        clabel='% models',
    )

    x0, x1, y0, y1 = SA_EXTENT
    slope_overlay = (slope_map * gf.coastline * gf.borders).opts(
        projection=ccrs.PlateCarree(),
        xlim=(x0, x1),
        ylim=(y0, y1),
    )
    agreement_overlay = (agreement_map * gf.coastline * gf.borders).opts(
        projection=ccrs.PlateCarree(),
        xlim=(x0, x1),
        ylim=(y0, y1),
    )

    return pn.Column(
        pn.pane.Markdown(f'#### Spatial trends · {variable_label} · {scen_label} · {window_label}'),
        pn.Row(slope_overlay, agreement_overlay, sizing_mode='stretch_width'),
        sizing_mode='stretch_width',
    )


def _update_scenario_visibility(window_key):
    w41_scenario.disabled = window_key == 'historical_1950_2014'
    return None


pn.bind(_update_scenario_visibility, window_key=w41_window, watch=True)

spatial_maps_pane = pn.bind(
    build_spatial_maps_panel,
    variable=w41_variable,
    window_key=w41_window,
    scenario=w41_scenario,
)

pn.Column(
    pn.Row(
        pn.Column('**Variable**', w41_variable, width=240),
        pn.Column('**Window**', w41_window, width=460),
        pn.Column('**Scenario**', w41_scenario, width=360),
        sizing_mode='stretch_width',
    ),
    pn.pane.Markdown('---'),
    spatial_maps_pane,
    sizing_mode='stretch_width',
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'4f70e165-8411-4e77-b654-b8a3f77d83fe': {'version…

---

## Section 5 — T-H decoupling

Spearman correlation between monthly anomalies of tas and hurs. Weakening across periods is interpreted as regional hydrological regime change. Continental aggregate is intentionally excluded — decoupling is a regional phenomenon and continental averaging would mask contrasting regimes.

### 5.1 T-H scatter: Baseline vs future

In [14]:
BASELINE_PERIOD = (1960, 2014)

w51_subregion = pn.widgets.Select(name='Subregion', options=SUBREGIONS_MONTHLY, value='Central Amazonia' if 'Central Amazonia' in SUBREGIONS_MONTHLY else SUBREGIONS_MONTHLY[0], width=280)
FUTURE_PERIODS_51 = {'Mid-century (2040–2069)': ('mid', (2040, 2069)), 'End-century (2070–2099)': ('end', (2070, 2099))}
w51_future_period = pn.widgets.RadioBoxGroup(name='Future period', options=FUTURE_PERIODS_51, value=('end', (2070, 2099)), inline=True)
w51_future_scenario = pn.widgets.Select(name='Future scenario', options={SCENARIO_LABELS[s]: s for s in ['ssp126', 'ssp245', 'ssp370', 'ssp585']}, value='ssp585', width=320)
w51_models = pn.widgets.MultiChoice(name='GCMs', options=MODELS, value=MODELS, width=520)

def _ols_line(x, y):
    if len(x) < 2:
        return np.array([]), np.array([])
    slope, intercept = np.polyfit(x, y, 1)
    xr_ = np.array([x.min(), x.max()])
    return xr_, slope * xr_ + intercept

def build_th_scatter(subregion, future_period, future_scenario, models):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    _, future_range = future_period
    tas_b, hurs_b = al.build_monthly_paired_th(df_monthly=df_monthly, scenario='historical', subregion=subregion, year_range=BASELINE_PERIOD, selected_models=models, use_ensemble_median=True, use_anomalies=True)
    tas_f, hurs_f = al.build_monthly_paired_th(df_monthly=df_monthly, scenario=future_scenario, subregion=subregion, year_range=future_range, selected_models=models, use_ensemble_median=True, use_anomalies=True)
    if len(tas_b) < 10 or len(tas_f) < 10:
        return pn.pane.Markdown('⚠️ Insufficient data for one period.')
    corr_b = al.compute_all_correlations(tas_b, hurs_b)
    corr_f = al.compute_all_correlations(tas_f, hurs_f)
    df_b = pd.DataFrame({'tas_anom': tas_b, 'hurs_anom': hurs_b})
    df_f_label = f'{SCENARIO_LABELS[future_scenario].split(" (")[0]} · {future_range[0]}–{future_range[1]}'
    df_f = pd.DataFrame({'tas_anom': tas_f, 'hurs_anom': hurs_f})
    sc_b = df_b.hvplot.scatter(x='tas_anom', y='hurs_anom', color='#1f77b4', size=18, alpha=0.55, label='Baseline 1960–2014')
    sc_f = df_f.hvplot.scatter(x='tas_anom', y='hurs_anom', color='#d62728', size=18, alpha=0.55, label=df_f_label)
    xb, yb = _ols_line(tas_b, hurs_b)
    xf, yf = _ols_line(tas_f, hurs_f)
    line_b = hv.Curve((xb, yb)).opts(color='#1f77b4', line_dash='dashed', line_width=2)
    line_f = hv.Curve((xf, yf)).opts(color='#d62728', line_dash='dashed', line_width=2)
    overlay = (sc_b * sc_f * line_b * line_f).opts(
        title=f'T-H scatter · {subregion} · ensemble median',
        xlabel='tas anomaly (°C)', ylabel='hurs anomaly (%)',
        width=720, height=480, legend_position='top_right', show_grid=True, toolbar='above',
    )
    rel_change = 100 * (abs(corr_f['spearman_r']) - abs(corr_b['spearman_r'])) / abs(corr_b['spearman_r'])
    stats_md = f"""
**Baseline 1960–2014**
ρ Spearman = **{corr_b['spearman_r']:+.3f}**
CI 95% = [{corr_b['spearman_lo95']:+.3f}, {corr_b['spearman_hi95']:+.3f}]
n = {corr_b['n']}

**{df_f_label}**
ρ Spearman = **{corr_f['spearman_r']:+.3f}**
CI 95% = [{corr_f['spearman_lo95']:+.3f}, {corr_f['spearman_hi95']:+.3f}]
n = {corr_f['n']}

---

**Relative change:** {rel_change:+.1f}% in absolute magnitude
"""
    return pn.Row(pn.pane.HoloViews(overlay), pn.pane.Markdown(stats_md, width=280))

scatter_pane = pn.bind(build_th_scatter, subregion=w51_subregion, future_period=w51_future_period, future_scenario=w51_future_scenario, models=w51_models)

pn.Column(
    pn.Row(pn.Column('**Subregion**', w51_subregion, width=300), pn.Column('**Future scenario**', w51_future_scenario, width=340)),
    pn.Column('**Future period**', w51_future_period, '**GCMs**', w51_models),
    pn.pane.Markdown('---'),
    scatter_pane,
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'0339b0a1-df8a-467d-8a05-a18c5601e0ec': {'version…

### 5.2 Evolving correlation table

Spearman ρ across subregions, periods and scenarios. ★ marks weakening >50% vs baseline.

In [15]:
w52_models = pn.widgets.MultiChoice(name='GCMs', options=MODELS, value=MODELS, width=520)

EVO_PERIODS = [
    ('Baseline 1960–2014', 'historical', (1960, 2014)),
    ('Mid SSP2-4.5',       'ssp245',     (2040, 2069)),
    ('Mid SSP5-8.5',       'ssp585',     (2040, 2069)),
    ('End SSP2-4.5',       'ssp245',     (2070, 2099)),
    ('End SSP5-8.5',       'ssp585',     (2070, 2099)),
]

def build_evo_corr_table(models):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    rows = []
    for subregion in SUBREGIONS_MONTHLY:
        row = {'Subregion': subregion}
        baseline_abs = None
        for col_label, scenario, year_range in EVO_PERIODS:
            tas, hurs = al.build_monthly_paired_th(df_monthly=df_monthly, scenario=scenario, subregion=subregion, year_range=year_range, selected_models=models, use_ensemble_median=True, use_anomalies=True)
            if len(tas) < 10:
                row[col_label] = ''
                continue
            r = al.compute_all_correlations(tas, hurs)['spearman_r']
            if 'Baseline' in col_label:
                baseline_abs = abs(r)
                row[col_label] = f'{r:+.3f}'
            else:
                tag = ' ★' if (baseline_abs is not None and abs(r) < 0.5 * baseline_abs) else ''
                row[col_label] = f'{r:+.3f}{tag}'
        rows.append(row)
    df_table = pd.DataFrame(rows)
    def extract_r(s):
        if not s:
            return np.nan
        try:
            return float(s.replace('★', '').strip())
        except ValueError:
            return np.nan
    numeric_cols = [c for c in df_table.columns if c != 'Subregion']
    def style_cell(val):
        r = extract_r(val)
        if not np.isfinite(r):
            return ''
        abs_r = abs(r)
        if r < 0:
            intensity = min(abs_r, 1.0)
            red = int(240 - 180 * intensity)
            green = int(220 - 120 * intensity)
            color = f'rgb({red},{green},{180 + int(75*intensity)})'
            text = 'white' if intensity > 0.5 else 'black'
            return f'background-color: {color}; color: {text}'
        else:
            intensity = min(abs_r, 1.0)
            color = f'rgb(220,{int(150 - 100*intensity)},{int(150 - 100*intensity)})'
            return f'background-color: {color}; color: white'
    styler = df_table.style.map(style_cell, subset=numeric_cols).set_properties(**{'text-align': 'center', 'font-weight': '600'})
    header = pn.pane.Markdown(f'**Spearman ρ (tas, hurs) · monthly anomalies · ensemble median**  \nEnsemble: {len(models)} / {len(MODELS)} models · ★ weakening >50% vs baseline')
    return pn.Column(header, pn.pane.DataFrame(styler, index=False, escape=False, sizing_mode='stretch_width'))

evo_table_pane = pn.bind(build_evo_corr_table, models=w52_models)

pn.Column(pn.Column('**GCMs**', w52_models), pn.pane.Markdown('---'), evo_table_pane)

BokehModel(combine_events=True, render_bundle={'docs_json': {'7f303465-f078-4c82-a840-04b4d6347033': {'version…

### 5.3 Quantitative decoupling indicator

A subregion is *decoupled* when |ρ_future| < threshold × |ρ_baseline|. Threshold is user-adjustable to probe robustness.

In [16]:
w53_threshold = pn.widgets.FloatSlider(name='Weakening threshold (%)', start=10, end=90, step=5, value=50, width=420)
w53_models = pn.widgets.MultiChoice(name='GCMs', options=MODELS, value=MODELS, width=520)

FUTURE_SCENARIOS_53 = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
END_CENTURY_RANGE = (2070, 2099)

def build_decoupling_indicator(threshold_pct, models):
    if not models:
        return pn.pane.Markdown('⚠️ Select at least one model.')
    threshold_frac = threshold_pct / 100.0
    rows = []
    for subregion in SUBREGIONS_MONTHLY:
        tas_b, hurs_b = al.build_monthly_paired_th(df_monthly=df_monthly, scenario='historical', subregion=subregion, year_range=BASELINE_PERIOD, selected_models=models, use_ensemble_median=True, use_anomalies=True)
        if len(tas_b) < 10:
            continue
        r_baseline = al.compute_all_correlations(tas_b, hurs_b)['spearman_r']
        if not np.isfinite(r_baseline) or abs(r_baseline) < 0.01:
            continue
        abs_baseline = abs(r_baseline)
        row = {'Subregion': subregion}
        for scenario in FUTURE_SCENARIOS_53:
            tas_f, hurs_f = al.build_monthly_paired_th(df_monthly=df_monthly, scenario=scenario, subregion=subregion, year_range=END_CENTURY_RANGE, selected_models=models, use_ensemble_median=True, use_anomalies=True)
            if len(tas_f) < 10:
                row[scenario] = ''
                continue
            r_f = al.compute_all_correlations(tas_f, hurs_f)['spearman_r']
            if not np.isfinite(r_f):
                row[scenario] = ''
                continue
            weakening = 1.0 - (abs(r_f) / abs_baseline)
            tag = 'DECOUPLED' if weakening > threshold_frac else 'coupled'
            sign = '+' if weakening >= 0 else ''
            row[scenario] = f'{sign}{100*weakening:.0f}% · {tag}'
        rows.append(row)
    df_table = pd.DataFrame(rows)
    df_table.columns = ['Subregion'] + [SCENARIO_LABELS[s].split(' (')[0] for s in FUTURE_SCENARIOS_53]
    scenario_cols = df_table.columns[1:].tolist()
    def style_cell(val):
        if not val:
            return ''
        if 'DECOUPLED' in val:
            try:
                pct = float(val.split('%')[0].replace('+', ''))
                intensity = min(pct / 100, 1.0)
            except (ValueError, IndexError):
                intensity = 0.7
            r = 200 + int(35 * intensity)
            g = int(60 - 30 * intensity)
            b = int(60 - 30 * intensity)
            return f'background-color: rgb({r},{g},{b}); color: white; font-weight: 700'
        return 'background-color: #d4edda; color: #155724; font-weight: 600'
    styler = df_table.style.map(style_cell, subset=scenario_cols).set_properties(**{'text-align': 'center'})
    header = pn.pane.Markdown(f'**Decoupling end-century (2070–2099) vs baseline (1960–2014)**  \nActive threshold: **{threshold_pct:.0f}%** weakening · Ensemble: {len(models)} / {len(MODELS)} models')
    return pn.Column(header, pn.pane.DataFrame(styler, index=False, escape=False, sizing_mode='stretch_width'))

decoupling_pane = pn.bind(build_decoupling_indicator, threshold_pct=w53_threshold, models=w53_models)

pn.Column(pn.Column('**Weakening threshold**', w53_threshold, '**GCMs**', w53_models), pn.pane.Markdown('---'), decoupling_pane)

BokehModel(combine_events=True, render_bundle={'docs_json': {'91dd6e25-fe2c-486c-9daa-f2aae5688e4b': {'version…

---

## Section 6 — Browser dashboard view

Integrated Panel app view for browser use. The final object is a `FastListTemplate` with a left sidebar and two tabs: **Trends** and **T-H Decoupling**.

To open it as a browser app from a terminal, run:

```bash
panel serve tablero2_geoviews_dashboard.ipynb --show --autoreload
```

In [17]:
wD_variable = pn.widgets.RadioBoxGroup(
    name='Variable',
    options={VARIABLE_LABELS[v]: v for v in VARIABLES},
    value='tas',
    inline=False,
)
wD_subregion = pn.widgets.Select(
    name='Subregion',
    options=SUBREGIONS,
    value='Central Amazonia' if 'Central Amazonia' in SUBREGIONS else SUBREGIONS[0],
    width=260,
)
wD_scenarios = pn.widgets.CheckButtonGroup(
    name='Scenarios',
    options={SCENARIO_LABELS[s].split(' (')[0]: s for s in SSP_ORDER if s in SCENARIOS},
    value=[s for s in SSP_ORDER if s in SCENARIOS],
    button_type='primary',
    button_style='outline',
)
wD_models = pn.widgets.MultiChoice(name='GCM ensemble', options=MODELS, value=MODELS, width=260)
wD_year_range = pn.widgets.IntRangeSlider(
    name='Time window',
    start=YEAR_MIN,
    end=YEAR_MAX,
    value=(YEAR_MIN, YEAR_MAX),
    step=1,
    width=260,
)
wD_mk_window = pn.widgets.RadioBoxGroup(
    name='MK / map window',
    options={
        'Historical': 'historical_1950_2014',
        'Projected': 'projected_2015_2100',
        'Full': 'full_1950_2100',
    },
    value='full_1950_2100',
    inline=False,
)
wD_map_scenario = pn.widgets.Select(
    name='Map scenario',
    options={SCENARIO_LABELS[s]: s for s in ['ssp126', 'ssp245', 'ssp370', 'ssp585']},
    value='ssp585',
    width=260,
)
wD_dec_threshold = pn.widgets.FloatSlider(
    name='Decoupling threshold (%)',
    start=10,
    end=90,
    step=5,
    value=50,
    width=260,
)

In [18]:
def D_build_timeseries(variable, subregion, scenarios, models, year_range):
    return build_timeseries_plot(variable, subregion, scenarios, models, year_range, show_band=True)


def D_build_mk_table(variable, mk_window, models):
    return build_mk_table_panel(variable, mk_window, models)


def D_build_hs(variable, subregion, models):
    return build_hawkins_sutton_plots(variable, subregion, models)


def D_build_maps(variable, mk_window, map_scenario):
    return build_spatial_maps_panel(variable, mk_window, map_scenario)


def D_build_scatter(subregion, scenarios, models):
    sub = subregion if subregion in SUBREGIONS_MONTHLY else SUBREGIONS_MONTHLY[0]
    future_scen = scenarios[-1] if scenarios and scenarios[-1] != 'historical' else 'ssp585'
    if future_scen == 'historical':
        future_scen = 'ssp585'
    return build_th_scatter(sub, ('end', (2070, 2099)), future_scen, models)


def D_build_evo_table(models):
    return build_evo_corr_table(models)


def D_build_decoupling(threshold, models):
    return build_decoupling_indicator(threshold, models)


def _update_dashboard_map_scenario_visibility(window_key):
    wD_map_scenario.disabled = window_key == 'historical_1950_2014'
    return None


pn.bind(_update_dashboard_map_scenario_visibility, window_key=wD_mk_window, watch=True)


tab_trends = pn.Column(
    pn.pane.Markdown('### Time series · ensemble bands p10–p90'),
    pn.bind(
        D_build_timeseries,
        variable=wD_variable,
        subregion=wD_subregion,
        scenarios=wD_scenarios,
        models=wD_models,
        year_range=wD_year_range,
    ),
    pn.pane.Markdown("### Mann-Kendall trends + Sen's slope"),
    pn.bind(D_build_mk_table, variable=wD_variable, mk_window=wD_mk_window, models=wD_models),
    pn.pane.Markdown('### Hawkins-Sutton uncertainty decomposition'),
    pn.bind(D_build_hs, variable=wD_variable, subregion=wD_subregion, models=wD_models),
    pn.pane.Markdown('### Interactive spatial trends'),
    pn.bind(D_build_maps, variable=wD_variable, mk_window=wD_mk_window, map_scenario=wD_map_scenario),
    sizing_mode='stretch_width',
)


tab_decoupling = pn.Column(
    pn.pane.Markdown('### T-H scatter · Baseline vs End-century'),
    pn.bind(D_build_scatter, subregion=wD_subregion, scenarios=wD_scenarios, models=wD_models),
    pn.pane.Markdown('### Evolving Spearman ρ table'),
    pn.bind(D_build_evo_table, models=wD_models),
    pn.pane.Markdown('### Quantitative decoupling indicator'),
    pn.bind(D_build_decoupling, threshold=wD_dec_threshold, models=wD_models),
    sizing_mode='stretch_width',
)


sidebar = pn.Column(
    pn.pane.Markdown('## Global controls'),
    '**Variable**', wD_variable,
    '**Subregion**', wD_subregion,
    '**Scenarios**', wD_scenarios,
    '**GCM ensemble**', wD_models,
    '**Time window**', wD_year_range,
    pn.pane.Markdown('---'),
    '**MK / map window**', wD_mk_window,
    '**Map scenario**', wD_map_scenario,
    '**Decoupling threshold**', wD_dec_threshold,
    width=300,
    sizing_mode='fixed',
)


tabs = pn.Tabs(
    ('Trends', tab_trends),
    ('T-H Decoupling', tab_decoupling),
    dynamic=True,
    sizing_mode='stretch_width',
)


app = pn.template.FastListTemplate(
    title='Climate Trends Dashboard — South America',
    sidebar=[sidebar],
    main=[tabs],
    main_max_width='1500px',
)

server = app.show(threaded=True)

Launching server at http://localhost:39693


In [19]:
# =============================================================================
# Section 6 — Browser dashboard app
# =============================================================================
# This cell only DEFINES the app. It does not display it.
# Important: do not leave `app` or `dashboard` as the last line here.
# That can register Bokeh models in the notebook document and later break app.show().

pn.config.sizing_mode = 'stretch_width'

ALL_VALUE = '__all__'


def _as_options_with_all(values):
    return {'All': ALL_VALUE, **{str(v): v for v in values}}


def _selected_models(model_value):
    if model_value == ALL_VALUE:
        return list(MODELS)
    return [model_value]


def _selected_scenarios(scenario_value):
    if scenario_value == ALL_VALUE:
        return [s for s in SSP_ORDER if s in SCENARIOS]
    return [scenario_value]


def _scenario_for_single_outputs(scenario_value):
    # Spatial maps and T-H scatter require one scenario.
    # If the user selects All, use SSP5-8.5 as the reference high-emissions scenario.
    if scenario_value == ALL_VALUE:
        return 'ssp585'
    return scenario_value


def _window_from_year_range(year_range):
    start, end = year_range

    if end <= 2014:
        return 'historical_1950_2014'

    if start >= 2015:
        return 'projected_2015_2100'

    return 'full_1950_2100'


def _window_label(window_key):
    labels = {
        'historical_1950_2014': 'Historical 1950–2014',
        'projected_2015_2100': 'Projected 2015–2100',
        'full_1950_2100': 'Full 1950–2100',
    }
    return labels.get(window_key, window_key)


def _th_period_range(period_key):
    if period_key == 'mid':
        return (2040, 2069)
    return (2070, 2099)


def _safe_panel(builder, height=None):
    try:
        obj = builder()
        return pn.panel(obj, sizing_mode='stretch_width', height=height)
    except Exception as exc:
        return pn.pane.Alert(
            f'Error while building this panel:\n\n{type(exc).__name__}: {exc}',
            alert_type='danger',
            sizing_mode='stretch_width',
        )


def _build_spatial_maps_compatible(variable, window_key, scenario):
    """
    Compatibility wrapper for the GeoViews/Bokeh spatial map function.

    The current build_spatial_maps_panel() version does not accept show_sig.
    This wrapper keeps the dashboard stable if the function signature changes later.
    """
    try:
        return build_spatial_maps_panel(
            variable=variable,
            window_key=window_key,
            scenario=scenario,
        )
    except TypeError:
        # Fallback for versions that may use positional arguments.
        return build_spatial_maps_panel(variable, window_key, scenario)


def make_dashboard_app():
    # -------------------------------------------------------------------------
    # Fresh widgets for each app.show() execution
    # -------------------------------------------------------------------------
    variable = pn.widgets.Select(
        name='Variable',
        options={VARIABLE_LABELS.get(v, v): v for v in VARIABLES},
        value='tas' if 'tas' in VARIABLES else VARIABLES[0],
        width=190,
    )

    scenario = pn.widgets.Select(
        name='Scenario',
        options={
            'All': ALL_VALUE,
            **{
                SCENARIO_LABELS[s].split(' (')[0]: s
                for s in SSP_ORDER
                if s in SCENARIOS
            },
        },
        value=ALL_VALUE,
        width=190,
    )

    model = pn.widgets.Select(
        name='GCM',
        options=_as_options_with_all(MODELS),
        value=ALL_VALUE,
        width=190,
    )

    subregion = pn.widgets.Select(
        name='Subregion',
        options=SUBREGIONS,
        value='Central Amazonia' if 'Central Amazonia' in SUBREGIONS else SUBREGIONS[0],
        width=190,
    )

    year_range = pn.widgets.IntRangeSlider(
        name='Time window',
        start=YEAR_MIN,
        end=YEAR_MAX,
        value=(YEAR_MIN, YEAR_MAX),
        step=1,
        width=190,
    )

    th_period = pn.widgets.Select(
        name='T-H period',
        options={
            'Mid-century 2040–2069': 'mid',
            'End-century 2070–2099': 'end',
        },
        value='end',
        width=190,
    )

    dec_threshold = pn.widgets.FloatSlider(
        name='Decoupling threshold (%)',
        start=10,
        end=90,
        step=5,
        value=50,
        width=190,
    )

    # -------------------------------------------------------------------------
    # Builders: always return fresh objects
    # -------------------------------------------------------------------------
    def build_timeseries_view(variable_value, subregion_value, scenario_value, model_value, years_value):
        return _safe_panel(
            lambda: build_timeseries_plot(
                variable=variable_value,
                subregion=subregion_value,
                scenarios=_selected_scenarios(scenario_value),
                models=_selected_models(model_value),
                year_range=years_value,
                show_band=True,
            ),
            height=320,
        )

    def build_mk_view(variable_value, model_value, years_value):
        window_key = _window_from_year_range(years_value)

        return pn.Column(
            pn.pane.Markdown(f"#### Mann-Kendall + Sen's slope · {_window_label(window_key)}"),
            _safe_panel(
                lambda: build_mk_table_panel(
                    variable=variable_value,
                    window_key=window_key,
                    models=_selected_models(model_value),
                )
            ),
            sizing_mode='stretch_width',
        )

    def build_hs_view(variable_value, subregion_value, model_value):
        return _safe_panel(
            lambda: build_hawkins_sutton_plots(
                variable=variable_value,
                subregion=subregion_value,
                models=_selected_models(model_value),
            ),
            height=310,
        )

    def build_map_view(variable_value, scenario_value, years_value):
        window_key = _window_from_year_range(years_value)
        map_scenario = _scenario_for_single_outputs(scenario_value)

        note = ''
        if scenario_value == ALL_VALUE and window_key != 'historical_1950_2014':
            note = ' · map scenario shown: SSP5-8.5'

        return pn.Column(
            pn.pane.Markdown(f"#### Trend map · {_window_label(window_key)}{note}"),
            _safe_panel(
                lambda: _build_spatial_maps_compatible(
                    variable=variable_value,
                    window_key=window_key,
                    scenario=map_scenario,
                ),
                height=590,
            ),
            sizing_mode='stretch_width',
        )

    def build_scatter_view(subregion_value, scenario_value, model_value, th_period_value):
        sub = subregion_value if subregion_value in SUBREGIONS_MONTHLY else SUBREGIONS_MONTHLY[0]
        future_scenario = _scenario_for_single_outputs(scenario_value)
        future_range = _th_period_range(th_period_value)

        return _safe_panel(
            lambda: build_th_scatter(
                subregion=sub,
                future_period=(th_period_value, future_range),
                future_scenario=future_scenario,
                models=_selected_models(model_value),
            ),
            height=430,
        )

    def build_evo_view(model_value):
        return _safe_panel(
            lambda: build_evo_corr_table(
                models=_selected_models(model_value),
            )
        )

    def build_decoupling_view(dec_threshold_value, model_value):
        return _safe_panel(
            lambda: build_decoupling_indicator(
                threshold_pct=dec_threshold_value,
                models=_selected_models(model_value),
            )
        )

    # -------------------------------------------------------------------------
    # Left global controls
    # -------------------------------------------------------------------------
    controls = pn.Column(
        pn.pane.Markdown('### Global controls'),
        variable,
        scenario,
        model,
        year_range,
        th_period,
        dec_threshold,
        width=220,
        styles={
            'border-right': '2px solid #1f4fbf',
            'padding': '12px',
            'background': '#ffffff',
        },
    )

    # -------------------------------------------------------------------------
    # Tab 1: Trends
    # -------------------------------------------------------------------------
    trends_left = pn.Column(
        pn.bind(
            build_map_view,
            variable_value=variable,
            scenario_value=scenario,
            years_value=year_range,
        ),
        sizing_mode='stretch_width',
    )

    trends_right = pn.Column(
        pn.pane.Markdown('#### Time series'),
        pn.bind(
            build_timeseries_view,
            variable_value=variable,
            subregion_value=subregion,
            scenario_value=scenario,
            model_value=model,
            years_value=year_range,
        ),
        pn.bind(
            build_mk_view,
            variable_value=variable,
            model_value=model,
            years_value=year_range,
        ),
        pn.pane.Markdown('#### Hawkins-Sutton uncertainty decomposition'),
        pn.bind(
            build_hs_view,
            variable_value=variable,
            subregion_value=subregion,
            model_value=model,
        ),
        sizing_mode='stretch_width',
    )

    trends_tab = pn.Column(
        pn.pane.Markdown('## Trend of the selected variable'),
        pn.Row(
            trends_left,
            trends_right,
            sizing_mode='stretch_width',
        ),
        sizing_mode='stretch_width',
    )

    # -------------------------------------------------------------------------
    # Tab 2: T-H decoupling
    # -------------------------------------------------------------------------
    decoupling_left = pn.Column(
        pn.pane.Markdown('#### T-H scatter'),
        pn.bind(
            build_scatter_view,
            subregion_value=subregion,
            scenario_value=scenario,
            model_value=model,
            th_period_value=th_period,
        ),
        sizing_mode='stretch_width',
    )

    decoupling_right = pn.Column(
        pn.pane.Markdown('#### T-H correlations'),
        pn.bind(
            build_evo_view,
            model_value=model,
        ),
        pn.pane.Markdown('#### Quantitative decoupling indicator'),
        pn.bind(
            build_decoupling_view,
            dec_threshold_value=dec_threshold,
            model_value=model,
        ),
        sizing_mode='stretch_width',
    )

    decoupling_tab = pn.Column(
        pn.pane.Markdown('## Temperature-humidity decoupling'),
        pn.Row(
            decoupling_left,
            decoupling_right,
            sizing_mode='stretch_width',
        ),
        sizing_mode='stretch_width',
    )

    tabs = pn.Tabs(
        ('Trends', trends_tab),
        ('Decoupling', decoupling_tab),
        dynamic=True,
        sizing_mode='stretch_width',
    )

    header = pn.pane.Markdown(
        '# Climate Trends Dashboard',
        sizing_mode='stretch_width',
        styles={
            'border-bottom': '2px solid #222222',
            'padding': '10px 16px',
            'margin-bottom': '8px',
        },
    )

    main = pn.Column(
        header,
        pn.Row(
            controls,
            pn.Column(
                tabs,
                sizing_mode='stretch_width',
                styles={'padding': '12px'},
            ),
            sizing_mode='stretch_width',
        ),
        sizing_mode='stretch_width',
    )

    return main

(node:174294) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)


In [ ]:
# =============================================================================
# Open dashboard in browser
# =============================================================================
# Esta celda sí abre el navegador.
# Si ya había un servidor activo desde una ejecución anterior, lo detenemos primero.

try:
    server.stop()
except NameError:
    pass
except Exception:
    pass

app = make_dashboard_app()

server = app.show(
    threaded=True,
    title='Climate Trends Dashboard — South America',
)

Launching server at http://localhost:45277


(node:174312) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
